##Chapter 6: Tools for Model Training and Experimenting/Project: 
* MultiClass classification for Computer Vision /Ch6-01-Training Base Model.py

In [0]:
!pip install -q pytorch-lightning==2.1.2 deltalake==0.14.0 deltatorch==0.0.3 evalidate==2.0.2 pillow==10.1.0
dbutils.library.restartPython()

In [0]:
import torch
import torchvision
import pytorch_lightning as pl
from torch import nn, optim, Tensor, manual_seed, argmax
from torch.autograd import Variable
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel
from torch.optim import Optimizer
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import Accuracy, MulticlassConfusionMatrix
from torchvision import transforms, models
from pytorch_lightning.utilities.model_summary import ModelSummary

import os
import numpy as np
import io
import logging
from math import ceil
from pyspark.sql.functions import col
from PIL import Image

train_df = (spark.read.format("delta")
            .load(train_delta_path)
            ).limit(10)

display(train_df)

display(dbutils.fs.ls(volume_model_path))

In [0]:
import mlflow
from mlia_utils import mlflow_funcs

experiment_path = f"/Users/{current_user}/intel-clf-training_action"
mlflow_funcs.mlflow_set_experiment(experiment_path) 
log_path = f"{volume_file_path}/intel_image_clf/intel_training_logger_board"
!mkdir {log_path}

In [0]:
class LitCVNet(pl.LightningModule):

        def __init__(self, num_classes = 6, learning_rate= 0.0001, family = "resnext", momentum = 0.1,  dropout_rate = 0):
            super().__init__()
            self.save_hyperparameters()
            self.family = family # we do not use familit explicitly, but you could play with 2 different family models using 1 script 
            self.momentum = momentum
            self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)
            self.learning_rate = learning_rate 
            self.model = self.get_model(num_classes)
            self.loss = nn.CrossEntropyLoss()
            self.confusion_matrix = MulticlassConfusionMatrix(num_classes=num_classes)
            self.test_pred = []  # collect predictions

        def get_model(self, num_classes):
            """
            This is the function that initialises our model.
            If we wanted to use other prebuilt model libraries like timm we would put that model here
            """
            backbone = torchvision.models.wide_resnet50_2(pretrained=True)
            for param in backbone.parameters():
                param.required_grad = False
            num_ftrt = backbone.fc.in_features
            backbone.fc = nn.Linear(num_ftrt, num_classes)
            return backbone

        # We do not overwrite our forward pass 
        def forward(self, x):
            x  = self.model(x)
            return x
        
        def training_step(self, batch, batch_idx):
            x = batch["content"]
            y = batch["label_id"]
            logits = self.forward(x)
            loss = self.loss(logits,y)
            # Track accuracy
            acc = self.accuracy(logits, y)
            # required format
            self.log("loss", torch.tensor([loss]), on_step=True, on_epoch=True, logger=True)
            self.log("acc", torch.tensor([acc]), on_step=True, on_epoch=True, logger=True)
            return  {"loss": loss, "acc": acc}
        
        def validation_step(self, batch, batch_idx):
            x = batch["content"]
            y = batch["label_id"]
            logits = self.forward(x)
            loss = self.loss(logits, y)
            acc = self.accuracy(logits, y)
            # required format
            self.log("val_loss", torch.tensor([loss]), prog_bar=True, on_step=True, on_epoch=True, logger=True)
            self.log("val_acc", torch.tensor([acc]), prog_bar=True, on_step=True, on_epoch=True, logger=True)
            return {"val_loss": loss, "val_acc": acc} 
        
        def test_step(self, batch, batch_idx):
            x = batch["content"]
            y = batch["label_id"]
            # Evaluate model
            logits = self.forward(x)
            # Track loss
            loss = self.loss(logits, y)
            self.log('test_loss', loss)
            # Track accuracy
            acc = self.accuracy(logits, y)
            self.log('test_accuracy', acc)
            # Collect predictions
            self.test_pred.extend(logits.cpu().numpy())
            # Update confusion matrix
            self.confusion_matrix.update(logits, y)

        # predict_step is optional unless you are doing some predictions
        def predict_step(self, batch, batch_idx, dataloader_idx=0):
            x = batch["content"]
            y = batch["label_id"]
            preds = self.forward(x)
            return preds
        
        def configure_optimizers(self):
            params = self.model.fc.parameters()
            optimizer = torch.optim.AdamW(params, lr=self.learning_rate)
            lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[4,6], gamma=0.06)
            # required format 
            return {"optimizer":optimizer, "lr_scheduler":lr_scheduler}


# COMMAND ----------

model = LitCVNet()
ModelSummary(model, max_depth=4)